# Elastodynamic MHM: spatial and temporal convergence

The local Newmark responses and global traction coupling implement Gomes et al. (2017), [Equations (42)–(48)](https://doi.org/10.20906/CPS/CILAMCE2017-0399). The cube study uses the analytical displacement in Equation (53), with $\lambda=\mu=0.4$, $\rho=1$, and zero initial displacement and velocity. Its source is derived independently from $\rho u_{tt}-\nabla\cdot\sigma(u)$.

The numerical settings are local P3 fields, linear vector face traces, two local subdivisions, $6n^3$ macro tetrahedra and $T=0.5$. The seven spatial levels use $n=1,2,3,4,5,6,8$ and $\Delta t=0.005$. The paper does not unambiguously specify the local degree or historical connectivity, so these declared choices do not identify identical historical matrices.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm import ElastodynamicStepper, TriangleMesh

root = Path.cwd()
data = root / "examples/results/elastodynamics"

## Small physical check

A constant body force on a free body produces an exact rigid translation. This inexpensive check verifies inertial units, time integration and all displacement/velocity components. It does not replace the spatial convergence study.

In [ ]:
mesh = TriangleMesh.unit_square()
acceleration = np.array([1.0, -0.4])
density, dt = 2.4, 0.02
with threadpool_limits(1), ElastodynamicStepper(
    mesh, time_step=dt, degree=2, density=density,
    traction={int(face): np.zeros(2) for face in mesh.boundary_faces},
) as stepper:
    stepper.initialize()
    for _ in range(4):
        solution = stepper.advance(density * acceleration)
    expected_u = 0.5 * solution.time**2 * acceleration
    expected_v = solution.time * acceleration
    assert solution.l2_error(expected_u) < 1e-11
    assert solution.l2_error(expected_v, velocity=True) < 1e-10
print("Rigid-translation balance verified")

## Persisted physical fields and norms

Every acquired field is checked against its recorded digest. Stress norms include the divergence inside each fine tetrahedron; this broken norm does not imply H(div)-conforming raw stress. The two quadrature rules used in the spatial campaign independently check the integrated errors.

In [ ]:
record = json.loads((data / "comparison.json").read_text())
assert [row["n"] for row in record["spatial"]] == [1, 2, 3, 4, 5, 6, 8]
for name, digest in record["field_sha256"].items():
    assert hashlib.sha256((data / name).read_bytes()).hexdigest() == digest
for row in record["spatial"]:
    assert row["quadrature_relative_change"] < 1e-10
    print(row["n"], row["macro_cells"], row["norms"])
print("Field digests and quadrature checks passed")

## Separate spatial and temporal errors

Spatial errors are measured against the analytical solution. Temporal differences use the same spatial space and a separately acquired $\Delta t=0.00009765625$ trajectory. Keeping these two comparisons separate prevents the spatial approximation floor from hiding the second-order time behavior. The slopes in the figure are references, not fitted error values.

In [ ]:
print("Time-reference refinement:", record["temporal_reference_refinement"])
temporal = record["temporal"]
assert len(temporal) == 8
steps = np.array([row["dt"] for row in temporal])
errors = np.array([row["difference_to_time_reference"]["displacement_l2"] for row in temporal])
print("Successive time orders:", np.log(errors[:-1] / errors[1:]) / np.log(steps[:-1] / steps[1:]))
display(Image(filename=str(root / "docs/figures/elastodynamics/convergence.png")))

## Displacement and velocity components

The section $z=0.37$ shows signed components, common analytical/numerical color limits and separate error scales. Lines are actual intersections of the macro tetrahedra with the section. The profile at $y=0.413,z=0.37$ retains independent one-sided polynomial values at fine and macro interfaces.

In [ ]:
for name in ("displacement-components", "velocity-components", "profiles"):
    display(Image(filename=str(root / "docs/figures/elastodynamics" / f"{name}.png")))

Independent DOLFINx/UFL full-system trajectories use the same P3/P1/Newmark spaces and Eq. (53) data. Nodal coordinates and permutations accompany the fields. These discrete comparisons complement the exact-solution convergence study.

In [ ]:
native = json.loads((root / "examples/results/elastodynamics/native/verification.json").read_text())
assert native["accepted"] and len(native["rows"]) == 5
[(row["n"], row["dt"], row["maximum_trajectory_difference"], row["physical_checks"]) for row in native["rows"]]

In [ ]:
for name in ("native-displacement", "native-velocity"):
    display(Image(filename=str(root / f"docs/figures/elastodynamics/{name}.png")))